In [5]:
import re
import numpy as np
from collections import Counter, defaultdict

train_file = "en_ewt-ud-train.txt"

def load_data(file):
    data = []
    words = []
    tags = []

    with open(file, encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                if words:
                    data.append((words, tags))
                    words, tags = [], []
                continue

            if line.startswith("#"):
                continue

            p = line.split("\t")

            if len(p) >= 4 and "-" not in p[0] and "." not in p[0]:
                words.append(p[1])
                tags.append(p[3])

    return data

data = load_data(train_file)

transition = defaultdict(Counter)
emission = defaultdict(Counter)
tag_count = Counter()
start = Counter()

for words, tags in data:
    start[tags[0]] += 1

    for word, tag in zip(words, tags):
        emission[tag][word] += 1
        tag_count[tag] += 1

    for i in range(1, len(tags)):
        transition[tags[i-1]][tags[i]] += 1

tags = list(tag_count)
vocab = set(word for tag in emission for word in emission[tag])

def viterbi(sentence):
    dp = [{}]
    path = {}

    for tag in tags:
        e = (emission[tag][sentence[0]] + 1) / (tag_count[tag] + len(vocab))
        s = (start[tag] + 1) / (len(data) + len(tags))
        dp[0][tag] = np.log(s) + np.log(e)

    for i in range(1, len(sentence)):
        dp.append({})
        for tag in tags:
            e = (emission[tag][sentence[i]] + 1) / (tag_count[tag] + len(vocab))
            best = max(
                (dp[i-1][prev] +
                 np.log((transition[prev][tag] + 1) /
                        (sum(transition[prev].values()) + len(tags))) +
                 np.log(e), prev)
                for prev in tags
            )
            dp[i][tag] = best[0]
            path[i, tag] = best[1]

    tag = max(dp[-1], key=dp[-1].get)
    result = [tag]

    for i in range(len(sentence)-1, 0, -1):
        tag = path[i, tag]
        result.append(tag)

    return result[::-1]

sentence = re.findall(r"\w+|[^\w\s]", input("Enter a sentence: "))
result = viterbi(sentence)

for word, tag in zip(sentence, result):
    print(word, "->", tag)

Enter a sentence:  i like ramen.


i -> PRON
like -> VERB
ramen -> NOUN
. -> PUNCT
